# 01. Diagnóstico inicial

Este notebook convierte y consolida los archivos crudos antes de perfilar su calidad. El paso 0 queda reservado para la descarga automática; aquí comienza el análisis del material ya disponible y los resultados del diagnóstico se guardan en `Data/Diagnosticos/`.

In [39]:
import csv
import re
from html.parser import HTMLParser
from pathlib import Path


def resolve_project_root() -> Path:
    marker = Path("Data") / "raw"
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"No se encontró {marker} a partir de {Path.cwd()}. "
        "Ejecute el notebook desde la raíz del repositorio o una subcarpeta que la contenga."
    )


ROOT = resolve_project_root()
DATOS_DIR = ROOT / "Data" / "raw"
CSV_DIR = ROOT / "Data" / "csv"
CONSOLIDATED_NAME = "EstablecimientosDiversificadoConsolidado.csv"

EXPECTED_HEADERS = [
    "CODIGO",
    "DISTRITO",
    "DEPARTAMENTO",
    "MUNICIPIO",
    "ESTABLECIMIENTO",
    "DIRECCION",
    "TELEFONO",
    "SUPERVISOR",
    "DIRECTOR",
    "NIVEL",
    "SECTOR",
    "AREA",
    "STATUS",
    "MODALIDAD",
    "JORNADA",
    "PLAN",
    "DEPARTAMENTAL",
]

CODE_PATTERN = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")


class _HTMLTableParser(HTMLParser):
    """Extrae filas de celdas <td> de cualquier tabla HTML.

    Los archivos .xls que entrega el portal del MINEDUC no son hojas de
    calculo binarias: son paginas HTML completas que el navegador guardaba
    con extension .xls.  No hace falta LibreOffice ni ninguna libreria
    de terceros para leerlos.
    """

    def __init__(self) -> None:
        super().__init__()
        self._rows: list[list[str]] = []
        self._current_row: list[str] = []
        self._current_cell: list[str] = []
        self._in_td: bool = False

    def handle_starttag(self, tag: str, attrs: list) -> None:
        if tag == "tr":
            self._current_row = []
        elif tag == "td":
            self._in_td = True
            self._current_cell = []

    def handle_endtag(self, tag: str) -> None:
        if tag == "tr":
            if self._current_row:
                self._rows.append(self._current_row)
        elif tag == "td":
            self._in_td = False
            # Colapsa espacios internos para eliminar saltos de linea y
            # espacios de no-separacion que el HTML suele introducir.
            cell_text = " ".join("".join(self._current_cell).split())
            self._current_row.append(cell_text)

    def handle_data(self, data: str) -> None:
        if self._in_td:
            self._current_cell.append(data)

    @property
    def rows(self) -> list[list[str]]:
        return self._rows


def parse_html_xls(xls_path: Path) -> list[list[str]]:
    """Lee un .xls del MINEDUC (HTML disfrazado) y devuelve todas las filas
    de celdas encontradas.  Usa encoding latin-1 porque el portal codifica
    los caracteres especiales del espanol (tildes, enes) en ISO-8859-1."""
    with xls_path.open("r", encoding="latin-1") as handle:
        html = handle.read()
    # Elimina bloques <script> que pueden confundir al parser de html.parser
    html = re.sub(
        r"<script\b[^<]*(?:(?!</script>)<[^<]*)*</script>",
        "",
        html,
        flags=re.IGNORECASE,
    )
    parser = _HTMLTableParser()
    parser.feed(html)
    return parser.rows


def find_header_index(rows: list[list[str]]) -> int:
    for index, row in enumerate(rows):
        if not row:
            continue
        first = row[0].strip().upper()
        if first == "CODIGO" and any(
            cell.strip().upper() == "ESTABLECIMIENTO" for cell in row
        ):
            return index
    raise ValueError("No se encontro la fila de encabezados esperada")


def extract_table(rows: list[list[str]]) -> list[list[str]]:
    """Filtra las filas de datos reales (codigo ##-##-####-##) y las
    normaliza al ancho esperado de EXPECTED_HEADERS."""
    header_index = find_header_index(rows)
    cleaned_rows: list[list[str]] = []
    for row in rows[header_index + 1 :]:
        if not row:
            continue
        if not CODE_PATTERN.match(row[0].strip()):
            continue
        normalized = [cell.strip() for cell in row[: len(EXPECTED_HEADERS)]]
        if len(normalized) < len(EXPECTED_HEADERS):
            normalized.extend([""] * (len(EXPECTED_HEADERS) - len(normalized)))
        cleaned_rows.append(normalized)
    return cleaned_rows


def write_csv(csv_path: Path, rows: list[list[str]]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(EXPECTED_HEADERS)
        writer.writerows(rows)


def main(force: bool = False) -> None:
    """Convierte los .xls a CSV y genera el consolidado.

    Parametros
    ----------
    force:
        Si es True, regenera todos los CSV aunque ya existan.
        Si es False (default), omite los archivos ya convertidos y solo
        regenera el consolidado si algun departamento cambio.
    """
    CSV_DIR.mkdir(parents=True, exist_ok=True)

    xls_files = sorted(DATOS_DIR.glob("*.xls"))
    if not xls_files:
        raise FileNotFoundError(
            f"No se encontraron archivos .xls en {DATOS_DIR}. "
            "Ejecute primero el notebook 00_descarga_datos.ipynb."
        )

    consolidated_rows: list[list[str]] = []
    summary: list[tuple[str, int, str]] = []  # (nombre, filas, estado)

    for xls_file in xls_files:
        dest_csv = CSV_DIR / f"{xls_file.stem}.csv"

        if dest_csv.exists() and not force:
            # Modo incremental: reutiliza el CSV ya generado
            with dest_csv.open(newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                next(reader)  # salta encabezado
                rows = list(reader)
            consolidated_rows.extend(rows)
            summary.append((xls_file.name, len(rows), "omitido (ya existe)"))
        else:
            raw_rows = parse_html_xls(xls_file)
            table_rows = extract_table(raw_rows)
            write_csv(dest_csv, table_rows)
            consolidated_rows.extend(table_rows)
            summary.append((xls_file.name, len(table_rows), "generado"))

    consolidated_path = CSV_DIR / CONSOLIDATED_NAME
    write_csv(consolidated_path, consolidated_rows)

    print("Archivos procesados:")
    total = 0
    for name, count, estado in summary:
        print(f"  {name}: {count:,} filas  [{estado}]")
        total += count
    print(f"Total consolidado: {total:,} filas")
    print(f"Salida principal:  {consolidated_path}")


main()

Archivos procesados:
  ALTA VERAPAZ.xls: 475 filas  [omitido (ya existe)]
  BAJA VERAPAZ.xls: 171 filas  [omitido (ya existe)]
  CHIMALTENANGO.xls: 435 filas  [omitido (ya existe)]
  CHIQUIMULA.xls: 239 filas  [omitido (ya existe)]
  CIUDAD CAPITAL.xls: 2,161 filas  [omitido (ya existe)]
  EL PROGRESO.xls: 158 filas  [omitido (ya existe)]
  ESCUINTLA.xls: 708 filas  [omitido (ya existe)]
  GUATEMALA.xls: 1,908 filas  [omitido (ya existe)]
  HUEHUETENANGO.xls: 591 filas  [omitido (ya existe)]
  IZABAL.xls: 413 filas  [omitido (ya existe)]
  JALAPA.xls: 183 filas  [omitido (ya existe)]
  JUTIAPA.xls: 392 filas  [omitido (ya existe)]
  PETEN.xls: 516 filas  [omitido (ya existe)]
  QUETZALTENANGO.xls: 551 filas  [omitido (ya existe)]
  QUICHE.xls: 322 filas  [omitido (ya existe)]
  RETALHULEU.xls: 364 filas  [omitido (ya existe)]
  SACATEPEQUEZ.xls: 430 filas  [omitido (ya existe)]
  SAN MARCOS.xls: 724 filas  [omitido (ya existe)]
  SANTA ROSA.xls: 213 filas  [omitido (ya existe)]
  SOLOL

In [40]:
import csv
import re
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

from IPython.display import display, Markdown


def resolve_project_root() -> Path:
    marker = Path('Data') / 'csv' / 'EstablecimientosDiversificadoConsolidado.csv'
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f'No se encontró {marker} a partir de {Path.cwd()}. '
        'Ejecute el notebook desde la raíz del repositorio (o una subcarpeta de esta).'
    )


base_dir = resolve_project_root()
csv_path = base_dir / 'Data' / 'csv' / 'EstablecimientosDiversificadoConsolidado.csv'
raw_dir = base_dir / 'Data' / 'Raw'
diagnostics_dir = base_dir / 'Data' / 'Diagnosticos'

with csv_path.open(newline='', encoding='utf-8', errors='replace') as handle:
    reader = csv.DictReader(handle)
    rows = [{key: (value or '').strip() for key, value in row.items()} for row in reader]

headers = reader.fieldnames or []
row_count = len(rows)
column_count = len(headers)
raw_department_catalog = {path.stem.strip().upper() for path in raw_dir.glob('*.xls')}

CATEGORICAL_COLUMNS = ['DEPARTAMENTO', 'MUNICIPIO', 'NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTAL']
FREE_TEXT_COLUMNS = ['ESTABLECIMIENTO', 'DIRECCION', 'SUPERVISOR', 'DIRECTOR']

EXPECTED_CATEGORICAL_DOMAINS = {
    'NIVEL': {'DIVERSIFICADO'},
    'AREA': {'URBANA', 'RURAL'},
}


PLACEHOLDERS = {'', '-', '--', '---', '----', 'SIN DATO', 'N/A', 'NA'}
CODE_PATTERN = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
PHONE_PATTERN = re.compile(r'^[0-9]+$')


def normalize_text(value: str) -> str:
    return re.sub(r'\s+', ' ', (value or '').strip())


def is_blank(value: str) -> bool:
    return normalize_text(value) == ''


def normalize_upper(value: str) -> str:
    return normalize_text(value).upper()


def value_counter(column_name: str) -> Counter:
    return Counter(normalize_text(row.get(column_name, '')) for row in rows)


def print_table(title: str, records: list[dict], limit: int = 10) -> None:
    display(Markdown(f'### {title}'))
    if not records:
        print('Sin resultados')
        return
    preview = records[:limit]
    columns = list(preview[0].keys())
    widths = {column: len(column) for column in columns}
    for record in preview:
        for column in columns:
            widths[column] = max(widths[column], len(str(record.get(column, ''))))
    print(' | '.join(column.ljust(widths[column]) for column in columns))
    print(' | '.join('-' * widths[column] for column in columns))
    for record in preview:
        print(' | '.join(str(record.get(column, '')).ljust(widths[column]) for column in columns))
    if len(records) > limit:
        print(f'... {len(records) - limit} filas adicionales')


def export_table_csv(records: list[dict], filename: str, fieldnames: list[str] | None = None) -> Path:
    """Exporta `records` a `Data/Diagnosticos/filename`. Si `records` viene vacío,
    `fieldnames` permite igual escribir el encabezado, para que un archivo con 0 filas
    se lea como "no hay casos que reportar" y no como "el reporte no llegó a correr"."""
    diagnostics_dir.mkdir(parents=True, exist_ok=True)
    path = diagnostics_dir / filename
    columns = fieldnames if fieldnames is not None else (list(records[0].keys()) if records else [])
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(records)
    return path


print(f'Ruta: {csv_path}')
print(f'Forma: {row_count:,} filas x {column_count} columnas')
print('Columnas:')
for header in headers:
    print(f'- {header}')
print('\nPrimeras 5 filas:')
pprint(rows[:5], width=140)
print(f'\nCatálogo local de departamentos detectado: {len(raw_department_catalog)} valores')

Ruta: d:\Usuarios\Javier\Escritorio\PRY1-DS\Data\csv\EstablecimientosDiversificadoConsolidado.csv
Forma: 11,867 filas x 17 columnas
Columnas:
- CODIGO
- DISTRITO
- DEPARTAMENTO
- MUNICIPIO
- ESTABLECIMIENTO
- DIRECCION
- TELEFONO
- SUPERVISOR
- DIRECTOR
- NIVEL
- SECTOR
- AREA
- STATUS
- MODALIDAD
- JORNADA
- PLAN
- DEPARTAMENTAL

Primeras 5 filas:
[{'AREA': 'URBANA',
  'CODIGO': '16-01-0137-46',
  'DEPARTAMENTAL': 'ALTA VERAPAZ',
  'DEPARTAMENTO': 'ALTA VERAPAZ',
  'DIRECCION': '6A. AVENIDA 1-15 ZONA 4',
  'DIRECTOR': '--',
  'DISTRITO': '16-006',
  'ESTABLECIMIENTO': 'INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN',
  'JORNADA': 'NOCTURNA',
  'MODALIDAD': 'MONOLINGUE',
  'MUNICIPIO': 'COBAN',
  'NIVEL': 'DIVERSIFICADO',
  'PLAN': 'DIARIO(REGULAR)',
  'SECTOR': 'PRIVADO',
  'STATUS': 'CERRADA DEFINITIVAMENTE',
  'SUPERVISOR': 'JORGE EDUARDO PAQUE LAZARO',
  'TELEFONO': ''},
 {'AREA': 'URBANA',
  'CODIGO': '16-01-0138-46',
  'DEPARTAMENTAL': 'ALTA VERAPAZ',
  'DEPARTAMENTO': 'ALTA VERAPA

## 1. Carga e inspección del conjunto de datos

Esta sección carga el consolidado de establecimientos educativos y deja visibles las primeras señales de calidad: tamaño, estructura, tipo de dato y valores únicos por variable, valores faltantes y duplicados.

### 1.1 Tipo de dato y valores únicos por variable

Las 17 variables llegan como texto desde el CSV crudo, tal como se descargó de MINEDUC. Aquí el equipo clasifica el tipo de dato lógico que le corresponde a cada una según su contenido observado (código, categórico o texto libre) y cuenta cuántos valores distintos tiene, para responder de forma explícita a los incisos 3.b y 3.d de la guía del proyecto.

In [41]:
def inferir_tipo_dato(column_name: str) -> str:
    if column_name == 'CODIGO':
        return 'codigo alfanumerico (patron fijo ##-##-####-##)'
    if column_name == 'DISTRITO':
        return 'codigo alfanumerico (formato variable: ##-### o ##-##-####)'
    if column_name == 'TELEFONO':
        return 'texto (digitos con separadores no uniformes; puede haber varios numeros por celda)'
    if column_name in CATEGORICAL_COLUMNS:
        return 'categorico (dominio cerrado)'
    if column_name in FREE_TEXT_COLUMNS:
        return 'texto libre'
    return 'texto'


tipo_valores_report = []
for column in headers:
    valores_normalizados = [normalize_text(row.get(column, '')) for row in rows]
    valores_no_vacios = [valor for valor in valores_normalizados if valor != '']
    contador = Counter(valores_no_vacios)
    valor_top, frecuencia_top = contador.most_common(1)[0] if contador else ('', 0)
    tipo_valores_report.append({
        'column': column,
        'tipo_dato_detectado': inferir_tipo_dato(column),
        'valores_unicos': len(contador),
        'valores_no_vacios': len(valores_no_vacios),
        'valor_mas_frecuente': valor_top,
        'frecuencia_top': frecuencia_top,
    })

print_table('Tipo de dato y valores únicos por variable', tipo_valores_report, limit=17)
tipo_valores_export = export_table_csv(
    tipo_valores_report,
    'ReporteTiposValoresUnicos.csv',
    fieldnames=list(tipo_valores_report[0].keys()),
)
print(f'Archivo exportado: {tipo_valores_export}')

### Tipo de dato y valores únicos por variable

column          | tipo_dato_detectado                                                                | valores_unicos | valores_no_vacios | valor_mas_frecuente                           | frecuencia_top
--------------- | ---------------------------------------------------------------------------------- | -------------- | ----------------- | --------------------------------------------- | --------------
CODIGO          | codigo alfanumerico (patron fijo ##-##-####-##)                                    | 11867          | 11867             | 16-01-0137-46                                 | 1             
DISTRITO        | codigo alfanumerico (formato variable: ##-### o ##-##-####)                        | 1681           | 11335             | 01-403                                        | 166           
DEPARTAMENTO    | categorico (dominio cerrado)                                                       | 23             | 11867             | CIUDAD CAPITAL                                | 

In [42]:
missing_report = []
for column in headers:
    missing_count = sum(1 for row in rows if is_blank(row.get(column, '')))
    missing_report.append({
        'column': column,
        'missing_count': missing_count,
        'missing_pct': round((missing_count / row_count) * 100, 2),
        'non_missing_count': row_count - missing_count,
    })

missing_report.sort(key=lambda record: (-record['missing_count'], record['column']))
print_table('Reporte de valores faltantes', missing_report, limit=17)
missing_export = export_table_csv(missing_report, 'ReporteFaltantes.csv')
print(f'Archivo exportado: {missing_export}')

### Reporte de valores faltantes

column          | missing_count | missing_pct | non_missing_count
--------------- | ------------- | ----------- | -----------------
DIRECTOR        | 1733          | 14.6        | 10134            
TELEFONO        | 946           | 7.97        | 10921            
SUPERVISOR      | 535           | 4.51        | 11332            
DISTRITO        | 532           | 4.48        | 11335            
DIRECCION       | 76            | 0.64        | 11791            
ESTABLECIMIENTO | 5             | 0.04        | 11862            
AREA            | 0             | 0.0         | 11867            
CODIGO          | 0             | 0.0         | 11867            
DEPARTAMENTAL   | 0             | 0.0         | 11867            
DEPARTAMENTO    | 0             | 0.0         | 11867            
JORNADA         | 0             | 0.0         | 11867            
MODALIDAD       | 0             | 0.0         | 11867            
MUNICIPIO       | 0             | 0.0         | 11867            
NIVEL     

In [43]:
full_duplicates = 0
seen_rows = set()
for row in rows:
    signature = tuple(row.get(column, '') for column in headers)
    if signature in seen_rows:
        full_duplicates += 1
    else:
        seen_rows.add(signature)

codigo_counter = Counter(row.get('CODIGO', '') for row in rows if not is_blank(row.get('CODIGO', '')))
codigo_duplicates = [
    {'CODIGO': codigo, 'repeticiones': count}
    for codigo, count in sorted(codigo_counter.items())
    if count > 1
]

invalid_codigo_rows = [
    {
        'row_number': index,
        'CODIGO': row.get('CODIGO', ''),
        'DISTRITO': row.get('DISTRITO', ''),
        'DEPARTAMENTO': row.get('DEPARTAMENTO', ''),
        'MUNICIPIO': row.get('MUNICIPIO', ''),
    }
    for index, row in enumerate(rows, start=2)
    if not CODE_PATTERN.match(row.get('CODIGO', ''))
]

duplicate_summary = [
    {'tipo': 'duplicados exactos', 'cantidad': full_duplicates},
    {'tipo': 'CODIGO repetido', 'cantidad': sum(item['repeticiones'] - 1 for item in codigo_duplicates)},
    {'tipo': 'CODIGO invalido', 'cantidad': len(invalid_codigo_rows)},
]

print_table('Resumen de duplicados e identificadores inválidos', duplicate_summary, limit=10)
print_table('Códigos repetidos', codigo_duplicates, limit=20)
print_table('Registros con CODIGO inválido', invalid_codigo_rows, limit=20)

duplicates_export = export_table_csv(
    codigo_duplicates, 'ReporteCodigosRepetidos.csv', fieldnames=['CODIGO', 'repeticiones']
)
invalid_codigo_export = export_table_csv(
    invalid_codigo_rows,
    'ReporteCodigosInvalidos.csv',
    fieldnames=['row_number', 'CODIGO', 'DISTRITO', 'DEPARTAMENTO', 'MUNICIPIO'],
)
print(f'Archivo exportado: {duplicates_export}')
print(f'Archivo exportado: {invalid_codigo_export}')
if not codigo_duplicates and not invalid_codigo_rows:
    print(
        'Hallazgo: CODIGO no tiene valores repetidos ni fuera del patrón ##-##-####-## '
        'en los 11,867 registros; por eso ambos reportes anteriores quedan con encabezado '
        'y cero filas de datos, no porque el diagnóstico haya fallado en correr.'
    )

### Resumen de duplicados e identificadores inválidos

tipo               | cantidad
------------------ | --------
duplicados exactos | 0       
CODIGO repetido    | 0       
CODIGO invalido    | 0       


### Códigos repetidos

Sin resultados


### Registros con CODIGO inválido

Sin resultados
Archivo exportado: d:\Usuarios\Javier\Escritorio\PRY1-DS\Data\Diagnosticos\ReporteCodigosRepetidos.csv
Archivo exportado: d:\Usuarios\Javier\Escritorio\PRY1-DS\Data\Diagnosticos\ReporteCodigosInvalidos.csv
Hallazgo: CODIGO no tiene valores repetidos ni fuera del patrón ##-##-####-## en los 11,867 registros; por eso ambos reportes anteriores quedan con encabezado y cero filas de datos, no porque el diagnóstico haya fallado en correr.


In [44]:
placeholder_rows = []
phone_rows = []
text_columns = ['SUPERVISOR', 'DIRECTOR', 'ESTABLECIMIENTO', 'DIRECCION']

for index, row in enumerate(rows, start=2):
    director = normalize_upper(row.get('DIRECTOR', ''))
    supervisor = normalize_upper(row.get('SUPERVISOR', ''))
    telefono = normalize_text(row.get('TELEFONO', ''))

    if director in PLACEHOLDERS:
        placeholder_rows.append({'row_number': index, 'column': 'DIRECTOR', 'value': row.get('DIRECTOR', '')})
    if supervisor in PLACEHOLDERS:
        placeholder_rows.append({'row_number': index, 'column': 'SUPERVISOR', 'value': row.get('SUPERVISOR', '')})

    if telefono and (not PHONE_PATTERN.match(telefono) or len(telefono) < 7 or len(telefono) > 12):
        phone_rows.append({'row_number': index, 'TELEFONO': telefono, 'length': len(telefono), 'has_digits_only': PHONE_PATTERN.match(telefono) is not None})

placeholder_summary = [
    {'column': 'DIRECTOR', 'placeholder_rows': sum(1 for item in placeholder_rows if item['column'] == 'DIRECTOR')},
    {'column': 'SUPERVISOR', 'placeholder_rows': sum(1 for item in placeholder_rows if item['column'] == 'SUPERVISOR')},
    {'column': 'TELEFONO', 'placeholder_rows': sum(1 for row in rows if normalize_upper(row.get('TELEFONO', '')) in PLACEHOLDERS)},
]

print_table('Resumen de placeholders y vacíos', placeholder_summary, limit=10)
print_table('Filas con placeholders en nombre', placeholder_rows, limit=20)
print_table('Teléfonos con formato sospechoso', phone_rows, limit=20)

placeholder_export = export_table_csv(placeholder_rows, 'ReportePlaceholdersNombres.csv')
phone_export = export_table_csv(phone_rows, 'ReporteTelefonosSospechosos.csv')
print(f'Archivo exportado: {placeholder_export}')
print(f'Archivo exportado: {phone_export}')

### Resumen de placeholders y vacíos

column     | placeholder_rows
---------- | ----------------
DIRECTOR   | 2024            
SUPERVISOR | 535             
TELEFONO   | 946             


### Filas con placeholders en nombre

row_number | column     | value
---------- | ---------- | -----
2          | DIRECTOR   | --   
54         | DIRECTOR   | --   
82         | DIRECTOR   |      
83         | DIRECTOR   |      
84         | DIRECTOR   |      
85         | DIRECTOR   |      
86         | DIRECTOR   |      
92         | DIRECTOR   |      
93         | DIRECTOR   | --   
97         | DIRECTOR   |      
98         | DIRECTOR   |      
100        | DIRECTOR   |      
104        | DIRECTOR   | ---  
105        | DIRECTOR   |      
110        | DIRECTOR   |      
110        | SUPERVISOR |      
112        | DIRECTOR   |      
112        | SUPERVISOR |      
116        | DIRECTOR   | ---  
121        | DIRECTOR   |      
... 2539 filas adicionales


### Teléfonos con formato sospechoso

row_number | TELEFONO                   | length | has_digits_only
---------- | -------------------------- | ------ | ---------------
328        | 78208583-78209143          | 17     | False          
485        | 79540830-79540909          | 17     | False          
486        | 79540830-79540909          | 17     | False          
488        | 79540830-79540909          | 17     | False          
655        | 78391288-78392217          | 17     | False          
671        | 783928                     | 6      | True           
674        | 79649696-78739432          | 17     | False          
676        | 78739432-79649696          | 17     | False          
677        | 78739432-79649696          | 17     | False          
746        | 78739432-79649696          | 17     | False          
772        | 78393245-78393246          | 17     | False          
789        | 78393245-78393246          | 17     | False          
794        | 78391332-78395850-78395997 | 26     | False      

## 2. Diagnóstico inicial de calidad de datos

Aquí se resumen los problemas más visibles por variable: faltantes, formatos inválidos, dominios inconsistentes y variaciones textuales que deberán normalizarse.

In [45]:
categorical_columns = CATEGORICAL_COLUMNS
category_summary = []

for column in categorical_columns:
    normalized_values = [normalize_upper(row.get(column, '')) for row in rows]
    counts = Counter(value for value in normalized_values if value)
    category_summary.append({
        'column': column,
        'unique_values': len(counts),
        'top_value': counts.most_common(1)[0][0] if counts else '',
        'top_count': counts.most_common(1)[0][1] if counts else 0,
    })

print_table('Resumen de categorías', category_summary, limit=20)

validation_rows = []
for column, expected_values in EXPECTED_CATEGORICAL_DOMAINS.items():
    observed = sorted({normalize_upper(row.get(column, '')) for row in rows if normalize_upper(row.get(column, ''))})
    unexpected = [value for value in observed if value not in expected_values]
    validation_rows.append({
        'column': column,
        'expected_values': ', '.join(sorted(expected_values)),
        'unexpected_values': ', '.join(unexpected),
        'unexpected_count': len(unexpected),
    })

invalid_departamentos = sorted({
    normalize_upper(row.get('DEPARTAMENTO', ''))
    for row in rows
    if normalize_upper(row.get('DEPARTAMENTO', '')) and normalize_upper(row.get('DEPARTAMENTO', '')) not in raw_department_catalog
})

invalid_departamentales = sorted({
    normalize_upper(row.get('DEPARTAMENTAL', ''))
    for row in rows
    if normalize_upper(row.get('DEPARTAMENTAL', '')) and normalize_upper(row.get('DEPARTAMENTAL', '')) not in raw_department_catalog
})

validation_rows.extend([
    {
        'column': 'DEPARTAMENTO',
        'expected_values': f'{len(raw_department_catalog)} departamentos locales',
        'unexpected_values': ', '.join(invalid_departamentos),
        'unexpected_count': len(invalid_departamentos),
    },
    {
        'column': 'DEPARTAMENTAL',
        'expected_values': f'{len(raw_department_catalog)} departamentos locales',
        'unexpected_values': ', '.join(invalid_departamentales),
        'unexpected_count': len(invalid_departamentales),
    },
])

print_table('Validación de catálogos y dominios', validation_rows, limit=20)
validation_export = export_table_csv(validation_rows, 'ReporteValidacionCatalogos.csv')
print(f'Archivo exportado: {validation_export}')
print(
    'Nota: este diagnóstico no incluye un catálogo oficial de MUNICIPIO (solo compara '
    'DEPARTAMENTO/DEPARTAMENTAL contra los 23 archivos fuente disponibles localmente). '
    'Esa validación se completó después, durante la limpieza, contra el catálogo de '
    'INE/Wikipedia documentado en AvanceProyecto1.md.'
)

### Resumen de categorías

column        | unique_values | top_value       | top_count
------------- | ------------- | --------------- | ---------
DEPARTAMENTO  | 23            | CIUDAD CAPITAL  | 2161     
MUNICIPIO     | 352           | ZONA 1          | 868      
NIVEL         | 1             | DIVERSIFICADO   | 11867    
SECTOR        | 4             | PRIVADO         | 9891     
AREA          | 3             | URBANA          | 9461     
STATUS        | 5             | ABIERTA         | 6860     
MODALIDAD     | 2             | MONOLINGUE      | 11394    
JORNADA       | 6             | DOBLE           | 3772     
PLAN          | 13            | DIARIO(REGULAR) | 7452     
DEPARTAMENTAL | 26            | GUATEMALA NORTE | 1489     


### Validación de catálogos y dominios

column        | expected_values          | unexpected_values                                                                                                                                     | unexpected_count
------------- | ------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------
NIVEL         | DIVERSIFICADO            |                                                                                                                                                       | 0               
AREA          | RURAL, URBANA            | SIN ESPECIFICAR                                                                                                                                       | 1               
DEPARTAMENTO  | 23 departamentos locales |                                                                                                              

## 3. Construcción de tablas de calidad de datos

Las tablas exportadas en esta sección documentan, de forma reproducible, la evidencia que el equipo usó para sustentar el proceso de limpieza y las decisiones de normalización que se aplicaron más adelante.

## 4. Plan de limpieza por variable

El plan siguiente resume qué limpiar, cómo validarlo y qué resultado esperar para cada una de las 17 variables del archivo consolidado.

In [46]:
cleaning_plan = [
    {
        'variable': 'CODIGO',
        'issues': 'Formato inválido, faltantes y repetidos.',
        'transformations': 'Conservar como texto, limpiar espacios y validar patrón ##-##-####-##.',
        'validation_rules': 'Debe cumplir el patrón definido y no repetirse entre filas activas.',
        'expected_outcome': 'Identificador estable y apto para cruces.'
    },
    {
        'variable': 'DISTRITO',
        'issues': 'Valores truncados o vacíos.',
        'transformations': 'Normalizar texto y validar contra catálogo local de distritos.',
        'validation_rules': 'Sin espacios sobrantes ni códigos incompletos.',
        'expected_outcome': 'Distrito consistente y comparable.'
    },
    {
        'variable': 'DEPARTAMENTO',
        'issues': 'Variantes de escritura y validación pendiente contra catálogo externo.',
        'transformations': 'Pasar a mayúsculas y comparar con catálogo local de referencia.',
        'validation_rules': 'Debe pertenecer al conjunto de departamentos detectado en los archivos fuente.',
        'expected_outcome': 'Departamentos homologados.'
    },
    {
        'variable': 'MUNICIPIO',
        'issues': 'Posibles variantes ortográficas y catálogo oficial no disponible en este diagnóstico.',
        'transformations': 'Normalizar mayúsculas, espacios y tildes; validar con catálogo oficial externo.',
        'validation_rules': 'Sin categorías nuevas tras la homologación.',
        'expected_outcome': 'Municipios listos para validación final.'
    },
    {
        'variable': 'ESTABLECIMIENTO',
        'issues': 'Comillas, tildes, espacios múltiples y variantes ortográficas.',
        'transformations': 'Limpiar caracteres invisibles y homogeneizar formato textual sin renombrar por criterio propio.',
        'validation_rules': 'Conservar el nombre institucional correcto.',
        'expected_outcome': 'Nombre del centro educativo estable.'
    },
    {
        'variable': 'DIRECCION',
        'issues': 'Abreviaturas, puntuación irregular y vacíos.',
        'transformations': 'Normalizar espacios, puntuación y abreviaturas evidentes.',
        'validation_rules': 'No debe quedar vacía ni con placeholders.',
        'expected_outcome': 'Dirección legible y uniforme.'
    },
    {
        'variable': 'TELEFONO',
        'issues': 'Faltantes, separadores inconsistentes, letras y longitudes raras.',
        'transformations': 'Conservar como texto y limpiar separadores; aislar teléfonos múltiples.',
        'validation_rules': 'Solo dígitos o valores explícitamente vacíos.',
        'expected_outcome': 'Contacto telefónico utilizable.'
    },
    {
        'variable': 'SUPERVISOR',
        'issues': 'Acentos, mayúsculas y espacios múltiples.',
        'transformations': 'Normalizar texto y eliminar placeholders.',
        'validation_rules': 'Nombre de persona sin valores genéricos.',
        'expected_outcome': 'Supervisor consistente.'
    },
    {
        'variable': 'DIRECTOR',
        'issues': 'Muchos placeholders y faltantes.',
        'transformations': 'Normalizar texto, detectar placeholders y dejar vacío solo cuando no exista dato real.',
        'validation_rules': 'No usar --, --- o SIN DATO como nombres válidos.',
        'expected_outcome': 'Director depurado.'
    },
    {
        'variable': 'NIVEL',
        'issues': 'Dominio reducido pero debe confirmarse.',
        'transformations': 'Homologar categorías al catálogo esperado.',
        'validation_rules': 'Solo valores permitidos por la fuente.',
        'expected_outcome': 'Nivel coherente.'
    },
    {
        'variable': 'SECTOR',
        'issues': 'Categorías potencialmente diversas o variantes de escritura.',
        'transformations': 'Estandarizar nombres equivalentes.',
        'validation_rules': 'Valores dentro del dominio permitido.',
        'expected_outcome': 'Sector uniforme.'
    },
    {
        'variable': 'AREA',
        'issues': 'Debe limitarse a urbano/rural.',
        'transformations': 'Normalizar mayúsculas y tildes.',
        'validation_rules': 'Solo URBANA o RURAL.',
        'expected_outcome': 'Área validada.'
    },
    {
        'variable': 'STATUS',
        'issues': 'Estados textuales con posible variación de escritura.',
        'transformations': 'Homologar a lista controlada de estados.',
        'validation_rules': 'No introducir estados nuevos sin soporte de fuente.',
        'expected_outcome': 'Estado institucional consistente.'
    },
    {
        'variable': 'MODALIDAD',
        'issues': 'Categorías compactas pero deben revisarse.',
        'transformations': 'Normalizar texto y revisar dominios.',
        'validation_rules': 'Conjunto controlado sin variantes innecesarias.',
        'expected_outcome': 'Modalidad depurada.'
    },
    {
        'variable': 'JORNADA',
        'issues': 'Categorías de horario con posibles variantes.',
        'transformations': 'Estandarizar etiquetas y revisar valores raros.',
        'validation_rules': 'Solo etiquetas oficiales de jornada.',
        'expected_outcome': 'Jornada uniforme.'
    },
    {
        'variable': 'PLAN',
        'issues': 'Valores equivalentes escritos de forma distinta o incompleta.',
        'transformations': 'Normalizar formato textual y consolidar equivalencias.',
        'validation_rules': 'Solo planes admitidos por el dataset.',
        'expected_outcome': 'Plan académico claro.'
    },
    {
        'variable': 'DEPARTAMENTAL',
        'issues': 'Debe coincidir con el departamento de referencia o el catálogo oficial.',
        'transformations': 'Comparar contra catálogo local y validar equivalencias.',
        'validation_rules': 'Sin discrepancias con el valor departamental esperado.',
        'expected_outcome': 'Campo territorial consistente.'
    },
]

print_table('Plan de limpieza por variable', cleaning_plan, limit=17)
cleaning_export = export_table_csv(cleaning_plan, 'ReportePlanLimpieza.csv')
print(f'Archivo exportado: {cleaning_export}')

### Plan de limpieza por variable

variable        | issues                                                                                | transformations                                                                                 | validation_rules                                                               | expected_outcome                         
--------------- | ------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------ | -----------------------------------------
CODIGO          | Formato inválido, faltantes y repetidos.                                              | Conservar como texto, limpiar espacios y validar patrón ##-##-####-##.                          | Debe cumplir el patrón definido y no repetirse entre filas activas.            | Identificador estable y apto para cruces.
DISTRITO        | V

## 5. Conclusiones y próximos pasos

El equipo deja documentadas en esta sección las tablas base del diagnóstico —exportadas a `Data/Diagnosticos/`— y separa con claridad lo que ya puede corregirse de lo que todavía depende de catálogos externos, sobre todo en MUNICIPIO.

Hallazgos principales:

- **CODIGO** no presenta valores repetidos ni fuera del patrón `##-##-####-##` en los 11,867 registros; por eso `ReporteCodigosRepetidos.csv` y `ReporteCodigosInvalidos.csv` quedan con encabezado y cero filas de datos, no porque el diagnóstico haya fallado en correr.
- **DIRECTOR** es la variable con más faltantes (14.6%), seguida de **TELEFONO** (7.97%) y **SUPERVISOR** (4.51%).
- **MUNICIPIO** no se pudo validar contra un catálogo oficial en este diagnóstico por no tener una fuente externa disponible en ese momento; esa validación se completó después, durante la limpieza, contra el catálogo de INE/Wikipedia documentado en `AvanceProyecto1.md`.
- **AREA** y **DEPARTAMENTAL** muestran valores fuera del dominio esperado inicialmente (`SIN ESPECIFICAR` y las particiones de Guatemala/Quiché, respectivamente); ambos se explican y documentan en el plan de limpieza en vez de tratarse como error.

Estas tablas alimentan directamente el plan de limpieza ejecutado por el equipo y documentado en `AvanceProyecto1.md`.